# Evaluation Tracer

The `evaluation.py` module defines a tracer that applies LangSmith run evaluators whenever completed root runs are persisted.

Evaluations may execute synchronously or through a thread-pool executor. Generated evaluation feedback is recorded through the configured LangSmith client, and evaluation results are grouped by target run and reference example.

### Functions

1. `wait_for_all_evaluators`: Waits for all evaluator tasks belonging to currently registered evaluator callback handlers to complete.

   Each live handler registered by this module has its `wait_for_futures` method called.

   * **Syntax:**
     ```python
     wait_for_all_evaluators() -> None
     ```

# EvaluatorCallbackHandler

`EvaluatorCallbackHandler` is a synchronous tracer that applies one or more LangSmith run evaluators whenever a root run is persisted.

The handler can use the shared LangChain executor, create a dedicated thread pool, or run evaluations synchronously. It may associate persisted runs with a reference example and can skip runs that do not contain outputs.

Evaluation feedback is sent through the configured LangSmith client. Logged results are stored by target run identifier and reference-example identifier.

## Bases

- `BaseTracer`

## Attributes

1. `name`: Stores the callback handler's identifying name.
   * **Type:**
     ```python
     name: str = "evaluator_callback_handler"
     ```

2. `example_id`: Stores the example identifier associated with evaluated runs.

   A string supplied to the constructor is converted into a `UUID`.

   * **Type:**
     ```python
     example_id: UUID | None = None
     ```

3. `client`: Stores the LangSmith client used to evaluate runs, read reference examples, and create feedback.
   * **Type:**
     ```python
     client: langsmith.Client
     ```

4. `evaluators`: Stores the sequence of evaluators applied to every persisted root run.
   * **Type:**
     ```python
     evaluators: Sequence[
         langsmith.RunEvaluator
     ] = ()
     ```

5. `executor`: Stores the thread-pool executor used to run evaluations.

   A value of `None` causes evaluator execution to occur synchronously.

   * **Type:**
     ```python
     executor: ThreadPoolExecutor | None = None
     ```

6. `futures`: Stores weak references to evaluation tasks submitted to an executor.
   * **Type:**
     ```python
     futures: weakref.WeakSet[
         Future[None]
     ] = weakref.WeakSet()
     ```

7. `skip_unfinished`: Controls whether persisted runs without outputs are skipped.
   * **Type:**
     ```python
     skip_unfinished: bool = True
     ```

8. `project_name`: Stores the LangSmith project used to organize evaluation-chain runs.

   The default project name is `"evaluators"`. A value of `None` disables the explicit evaluation project name.

   * **Type:**
     ```python
     project_name: str | None = None
     ```

9. `logged_eval_results`: Stores evaluation results grouped by target run identifier and reference-example identifier.
   * **Type:**
     ```python
     logged_eval_results: dict[
         tuple[
             str,
             str
         ],
         list[
             EvaluationResult
         ]
     ]
     ```

10. `lock`: Stores the thread lock used when updating `logged_eval_results`.
    * **Type:**
      ```python
      lock: threading.Lock
      ```

### Methods

1. `__init__`: Creates an evaluator callback handler.

   When `client` is omitted, the default LangChain LangSmith client is used. The `max_concurrency` value determines how evaluator work is executed:

   - `None` uses LangChain's shared executor.
   - A positive integer creates a dedicated `ThreadPoolExecutor`.
   - Zero or a negative integer disables the executor and runs evaluations synchronously.

   A dedicated executor is shut down automatically when the handler is finalized. The handler also registers itself with the module-level evaluator collection used by `wait_for_all_evaluators`.

   * **Syntax:**
     ```python
     __init__(
         self,
         evaluators: Sequence[
             langsmith.RunEvaluator
         ], # Evaluators applied to persisted root runs
         client: langsmith.Client | None = None, # LangSmith client used for evaluation and feedback
         example_id: UUID | str | None = None, # Reference example associated with evaluated runs
         skip_unfinished: bool = True, # Whether to skip runs without outputs
         project_name: str | None = "evaluators", # Project for evaluation-chain traces
         max_concurrency: int | None = None, # Maximum concurrent evaluator count
         **kwargs: Any # Additional BaseTracer arguments
     ) -> None
     ```

2. `_persist_run`: Applies every configured evaluator to a persisted root run.

   When `skip_unfinished` is enabled, a run without outputs is ignored. Otherwise, the run is copied, its `reference_example_id` is replaced with `example_id`, and each evaluator is executed synchronously or submitted to the configured executor.

   Submitted futures are added to `futures` so their completion can be awaited later.

   * **Syntax:**
     ```python
     _persist_run(
         self,
         run: Run # Completed root run to evaluate
     ) -> None
     ```

3. `wait_for_futures`: Waits until all currently tracked evaluator futures have completed.
   * **Syntax:**
     ```python
     wait_for_futures(
         self
     ) -> None
     ```

## Evaluation Result Handling

Each evaluator may return either one `EvaluationResult` or an `EvaluationResults` dictionary containing a `results` collection.

For every normalized result, the handler creates LangSmith feedback using its key, score, value, comment, correction, evaluator information, source-run identifier, and target-run identifier. Feedback is marked with the model feedback-source type.

The result is then appended to `logged_eval_results` under a key containing the target run identifier and the evaluated run's reference-example identifier. Updates to this mapping are protected by `lock`.